# 07 - Manual ReAct Agent

## Learning objective

Remove native function calling and implement an agent using only a text protocol, regex parsing, Python dispatch, and a scratchpad.

```text
Prompt text -> Thought/Action/Action Input -> regex -> Python tool -> Observation -> scratchpad
```

This notebook breaks the implementation into small, inspectable steps.

> **Quota note:** One complete run normally makes three Gemini requests.


## 1. Check the active Python kernel

The executable should point to the Conda environment used for notebooks, such as `nlp_env`.


In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 2. Optional package installation

If the course packages are not installed in the active kernel, remove the `#` and run the cell once. Then use **Kernel -> Restart Kernel**.


In [ ]:
# !pip install -r "../langchain-course/requirements.txt"


## 3. Corporate SSL and environment variables

`truststore` uses the Windows certificate store. The existing `.env` is loaded from the sibling script folder without printing secret values.


In [2]:
import os
from pathlib import Path

import truststore
from dotenv import load_dotenv

truststore.inject_into_ssl()

ENV_PATH = Path("../langchain-course/.env").resolve()
load_dotenv(dotenv_path=ENV_PATH)

print(".env exists:", ENV_PATH.exists())
print("Google key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))


.env exists: True
Google key loaded: True
Tavily key loaded: True
LangSmith key loaded: True
LangSmith tracing: true


## 4. Import parsing and inspection tools


In [3]:
import inspect
import re

from google import genai
from google.genai import types
from langsmith import traceable

MAX_ITERATIONS = 10
MODEL = "gemini-3.5-flash-lite" #"gemini-3.6-flash"
client = genai.Client()


## 5. Define regular Python tools

Arguments parsed from text initially arrive as strings, so `apply_discount` converts `price` to `float`.


In [4]:
@traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)


@traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier and return the final price.

    Available tiers: bronze, silver, gold.
    """
    price = float(price)
    discounts = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discounts.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)


tools = {
    "get_product_price": get_product_price,
    "apply_discount": apply_discount,
}


## 6. Test dictionary-based dynamic dispatch

`tools[tool_name]` retrieves the function. `*args` unpacks a list into positional arguments.


In [5]:
tool_name = "apply_discount"
args = ["1299.99", "gold"]

selected_tool = tools[tool_name]
observation = selected_tool(*args)

print("Selected function:", selected_tool)
print("Observation:", observation)


Selected function: <function apply_discount at 0x000001AFD4CDE5F0>
Observation: 1000.99


## 7. Generate tool descriptions with `inspect`

`__wrapped__` bypasses the `@traceable` wrapper so `inspect` sees the original signature.


In [6]:
def get_tool_descriptions(tools_dict):
    descriptions = []

    for tool_name, tool_function in tools_dict.items():
        original_function = getattr(
            tool_function,
            "__wrapped__",
            tool_function,
        )
        signature = inspect.signature(original_function)
        docstring = inspect.getdoc(original_function) or ""
        descriptions.append(f"{tool_name}{signature} - {docstring}")

    return "\n".join(descriptions)


tool_descriptions = get_tool_descriptions(tools)
tool_names = ", ".join(tools.keys())

print(tool_descriptions)
print("Allowed names:", tool_names)


get_product_price(product: str) -> float - Look up the price of a product in the catalog.
apply_discount(price: float, discount_tier: str) -> float - Apply a discount tier and return the final price.

Available tiers: bronze, silver, gold.
Allowed names: get_product_price, apply_discount


## 8. Build the ReAct prompt

Gemini receives no native tool configuration. All agency comes from this text protocol and the Python loop.


In [7]:
react_prompt = f"""
STRICT RULES: You must follow these exactly:
1. NEVER guess or assume any product price. You MUST call get_product_price first to get the real price.
2. Only call apply_discount AFTER you have received a price from get_product_price. Pass the exact price returned by get_product_price. Do NOT pass a made-up number.
3. NEVER calculate discounts yourself using math. Always use the apply_discount tool.
4. If the user does not specify a discount tier, ask them which tier to use. Do NOT assume one.

Answer the following questions as best you can. You have access to the following tools:

{tool_descriptions}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action, as comma separated values
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {{question}}
Thought:"""


print(react_prompt[:2000])


STRICT RULES: You must follow these exactly:
1. NEVER guess or assume any product price. You MUST call get_product_price first to get the real price.
2. Only call apply_discount AFTER you have received a price from get_product_price. Pass the exact price returned by get_product_price. Do NOT pass a made-up number.
3. NEVER calculate discounts yourself using math. Always use the apply_discount tool.
4. If the user does not specify a discount tier, ask them which tier to use. Do NOT assume one.

Answer the following questions as best you can. You have access to the following tools:

get_product_price(product: str) -> float - Look up the price of a product in the catalog.
apply_discount(price: float, discount_tier: str) -> float - Apply a discount tier and return the final price.

Available tiers: bronze, silver, gold.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [ge

## 9. Format the starting prompt


In [8]:
question = "What is the price of a laptop after applying a gold discount?"
prompt = react_prompt.format(question=question)
scratchpad = ""
full_prompt = prompt + scratchpad

print(full_prompt)



STRICT RULES: You must follow these exactly:
1. NEVER guess or assume any product price. You MUST call get_product_price first to get the real price.
2. Only call apply_discount AFTER you have received a price from get_product_price. Pass the exact price returned by get_product_price. Do NOT pass a made-up number.
3. NEVER calculate discounts yourself using math. Always use the apply_discount tool.
4. If the user does not specify a discount tier, ask them which tier to use. Do NOT assume one.

Answer the following questions as best you can. You have access to the following tools:

get_product_price(product: str) -> float - Look up the price of a product in the catalog.
apply_discount(price: float, discount_tier: str) -> float - Apply a discount tier and return the final price.

Available tiers: bronze, silver, gold.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [ge

## 10. Test the regex parser locally

This does not call Gemini.


In [9]:
sample_output = """I need the real laptop price.
Action: get_product_price
Action Input: laptop"""

action_match = re.search(r"Action:\s*(.+)", sample_output)
action_input_match = re.search(r"Action Input:\s*(.+)", sample_output)

print("Parsed action:", action_match.group(1).strip())
print("Parsed input:", action_input_match.group(1).strip())


Parsed action: get_product_price
Parsed input: laptop


## 11. Test argument parsing locally


In [10]:
tool_input_raw = "price=1299.99, discount_tier=gold"

raw_args = [value.strip() for value in tool_input_raw.split(",")]
args = [
    value.split("=", 1)[-1].strip().strip("'")
    for value in raw_args
]

print("Raw pieces:", raw_args)
print("Clean arguments:", args)


Raw pieces: ['price=1299.99', 'discount_tier=gold']
Clean arguments: ['1299.99', 'gold']


## 12. Define the traced plain-text Gemini call

No `tools` field is sent through the API. The stop sequence prevents Gemini from inventing an Observation.


In [11]:
def process_gemini_outputs(response):
    if response is None:
        return {
            "messages": [],
            "error": "Gemini returned no response",
        }

    return {
        "messages": [
            {
                "role": "assistant",
                "content": response.text or "",
            }
        ],
        "model_version": getattr(response, "model_version", None,),
        "response_id": getattr(response, "response_id", None,),
    }


@traceable(
    name="Gemini Generate Content",
    run_type="llm",
    process_outputs=process_gemini_outputs,
)
def gemini_chat_traced(model, messages, stop_sequences):
    contents = [
        types.Content(
            role=message["role"],
            parts=[types.Part(text=message["content"])],
        )
        for message in messages
    ]
    config = types.GenerateContentConfig(stop_sequences=stop_sequences)
    return client.models.generate_content(
        model=model,
        contents=contents,
        config=config,
    )


## 13. Optional one-step model output

> **This cell makes one Gemini request.**


In [12]:
one_step_response = gemini_chat_traced(
    model=MODEL,
    messages=[{"role": "user", "content": full_prompt}],
    stop_sequences=["\nObservation"],
)

print(one_step_response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Question: What is the price of a laptop after applying a gold discount?
Thought: I need to find the price of a laptop first using the get_product_price tool.
Action: get_product_price
Action Input: laptop


## 14. Define the complete ReAct loop


In [14]:
@traceable(name="Gemini ReAct Agent Loop")
def run_agent(question: str):
    prompt = react_prompt.format(question=question)
    scratchpad = ""

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"--- Iteration {iteration} ---")
        full_prompt = prompt + scratchpad

        response = gemini_chat_traced(
            model=MODEL,
            messages=[{"role": "user", "content": full_prompt}],
            stop_sequences=["\nObservation"],
        )
        output = response.text or ""
        print("LLM output:", output)

        final_match = re.search(r"Final Answer:\s*(.+)", output)
        if final_match:
            return final_match.group(1).strip(), scratchpad + output

        action_match = re.search(r"Action:\s*(.+)", output)
        input_match = re.search(r"Action Input:\s*(.+)", output)

        if not action_match or not input_match:
            raise ValueError("Could not parse Action and Action Input")

        tool_name = action_match.group(1).strip()
        tool_input_raw = input_match.group(1).strip()

        raw_args = [value.strip() for value in tool_input_raw.split(",")]
        args = [
            value.split("=", 1)[-1].strip().strip("'")
            for value in raw_args
        ]

        if tool_name not in tools:
            observation = (
                f"Error: unknown tool '{tool_name}'. "
                f"Available tools: {list(tools)}"
            )
        else:
            selected_tool = tools[tool_name]
            observation = str(selected_tool(*args))

        print("Tool:", tool_name)
        print("Arguments:", args)
        print("Observation:", observation)

        scratchpad += (
            f"{output}"
            f"Observation: {observation}"
            f"Thought:"
        )

    raise RuntimeError("Maximum iterations reached")


## 15. Run the ReAct agent

> **This cell normally makes three Gemini requests.**


In [15]:
final_answer, final_scratchpad = run_agent(question)

print("Final answer:", final_answer)


--- Iteration 1 ---
LLM output: Question: What is the price of a laptop after applying a gold discount?
Thought: I need to find the price of a laptop using the get_product_price tool first, according to the strict rules.
Action: get_product_price
Action Input: laptop
Tool: get_product_price
Arguments: ['laptop']
Observation: 1299.99
--- Iteration 2 ---
LLM output: Action: apply_discount
Action Input: 1299.99, gold
Tool: apply_discount
Arguments: ['1299.99', 'gold']
Observation: 1000.99
--- Iteration 3 ---
LLM output: Thought: I now know the final answer
Final Answer: The price of a laptop after applying a gold discount is $1000.99.
Final answer: The price of a laptop after applying a gold discount is $1000.99.


## 16. Inspect the complete scratchpad


In [16]:
print(final_scratchpad)

Question: What is the price of a laptop after applying a gold discount?
Thought: I need to find the price of a laptop using the get_product_price tool first, according to the strict rules.
Action: get_product_price
Action Input: laptopObservation: 1299.99Thought:Action: apply_discount
Action Input: 1299.99, goldObservation: 1000.99Thought:Thought: I now know the final answer
Final Answer: The price of a laptop after applying a gold discount is $1000.99.


## 17. Why ReAct is more fragile

Native function calling returns structured names and arguments. ReAct depends on exact text labels and parsing.

Potential failures include:

- Changed headings or punctuation
- Multiline action inputs
- Wrong tool names
- Wrong argument order
- Commas inside values
- Missing final-answer marker
- Model-generated fake observations without a stop sequence

## Key takeaways

- An agent can be built without native tool calling.
- The prompt defines the protocol.
- Regex interprets model text.
- Python validates and executes the tool.
- The scratchpad stores prior actions and observations.
- `tools[tool_name](*args)` performs dynamic positional dispatch.
- Native function calling is generally more reliable for production.

## Practice exercises

- Ask without a discount tier and inspect behavior.
- Add stricter regex anchors.
- Validate argument count against `inspect.signature()`.
- Compare the scratchpad with typed message history in Notebooks 05 and 06.
